In [3]:
import glob
import os 
import re
import copy
import json
import sys
import itertools
import random
import numpy as np
import pandas as pd

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

In [6]:
import transformers
print ("numpy: ", np.__version__)
print ("pandas: ", pd.__version__)
print ("transformers: ", transformers.__version__)

numpy:  2.4.3
pandas:  3.0.2
transformers:  5.8.0


In [1]:
path="./"

# 1. Loading Data

In [8]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv', index_col=0)
df_dc=df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9']].sum(axis=1)/df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9','WAXS_Int10','WAXS_Int11']].sum(axis=1)
df_param=pd.read_csv(path+'/data/params/param_sum.csv',index_col=0)
df_pred=pd.concat([df_param[['Property_Deg', 'Property_StrainBreak', 'Property_Young']], df_dc], axis=1)
df_pred['Property_Deg']=df_pred['Property_Deg']*100
df_pred['Property_StrainBreak']=df_pred['Property_StrainBreak']*100
df_pred['Property_Young']=df_pred['Property_Young']/1000
df_pred.columns=["Deg","Strain","Young","CrysDeg"]
df_pred["CrysDeg"]=df_pred["CrysDeg"]*100
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.8653,15.9800,1.533741,4.297316
18_1,8.6981,8.8500,1.595907,4.929744
19_1,8.1722,5.7200,1.704735,12.826432
21_1,7.8401,7.9833,1.549706,2.253836
22_1,8.7794,17.7067,1.499181,2.424432
...,...,...,...,...
78_3,0.9879,3.4867,1.853111,26.101993
79_3,1.0791,3.9867,1.812193,59.524230
81_3,8.0662,11.6667,1.587941,2.561380
82_3,7.8363,9.9367,1.442483,2.412784


In [9]:
df_pred2=df_pred.copy()
df_pred2=df_pred2.round(3)
df_pred2.columns=["Degradation rate", "Strain at break", "Young's modulus", "Crystallinity degree"]
df_pred2

,Degradation rate,Strain at break,Young's modulus,Crystallinity degree
17_1,7.865,15.980,1.534,4.297
18_1,8.698,8.850,1.596,4.930
19_1,8.172,5.720,1.705,12.826
21_1,7.840,7.983,1.550,2.254
22_1,8.779,17.707,1.499,2.424
...,...,...,...,...
78_3,0.988,3.487,1.853,26.102
79_3,1.079,3.987,1.812,59.524
81_3,8.066,11.667,1.588,2.561
82_3,7.836,9.937,1.442,2.413


# 2. Generation of Q&A sentences

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# Offline / pre-downloaded snapshot (this is what the original hard-coded):
# model_name = "./models/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,          # torch_dtype= is deprecated
    device_map="auto"
)

properties = [
    "Degradation rate",
    "Strain at break",
    "Young's modulus",
    "Crystallinity degree"
]

units = {
    "Degradation rate": "%",
    "Strain at break": "%",
    "Young's modulus": "GPa",
    "Crystallinity degree": "%"
}

def build_prompt(row, selected_props):

    prop_text = "\n".join([
        f"- {p}: {row[p]} {units[p]}"
        for p in selected_props
    ])

    props = ", ".join(selected_props)

    return f"""
Generate ONE QA pair for a polymer dataset.

--------------------------------
TASK
--------------------------------
You must generate:
- 1 question
- 1 answer

--------------------------------
STEP 1: Choose ONE question structure (IMPORTANT DIVERSITY RULE)
--------------------------------

Randomly choose ONE structure type:

A. Direct query
"What are the values of {props}?"

B. Simple factual
"What is the {props} of this polymer?"

C. Request style
"Please provide the values of {props}."

D. Inquiry style
"Could you provide the values of {props}?"

E. Open descriptive question
"What properties does this polymer exhibit in terms of {props}?"

F. Comparison-style framing
"How are the following properties characterized in this polymer: {props}?"

G. Scientific framing (non-list style)
"How do the material properties of this polymer relate to {props}?"

H. Neutral question
"Which values describe {props} in this polymer?"

IMPORTANT:
- Do NOT use phrases like "in what manner", "display", "characterize", "express"
- Do NOT always start with "What are the material properties"
- MUST vary sentence structure (not just wording)
- Avoid repetitive templates

--------------------------------
STEP 2: STRICT RULES
--------------------------------
- DO NOT include numbers and property values in the question, but Do include ALL selected property names in the question
- DO NOT include units in the question
- DO NOT add explanations in question
- DO NOT change property names in question and answer
- MUST include ALL selected property names in answer
- MUST NOT add or remove properties in question and answer

--------------------------------
STEP 3: ANSWER RULES
--------------------------------
- One sentence preferred
- List all properties clearly
- Include units
- No interpretation
- No adjectives like high/low/important
- No extra commentary
- DO NOT change property names

--------------------------------
PROPERTIES
--------------------------------
{prop_text}

--------------------------------
FORMAT
--------------------------------
Question: <question>
Answer: <answer>
""".strip()


def parse_qa(text):

    q_match = re.search(r"Question:\s*(.*)", text)
    # re.DOTALL added: without it a multi-line answer is truncated at the
    # first newline, and "Answer: ..." often wraps in practice.
    a_match = re.search(r"Answer:\s*(.*)", text, re.DOTALL)

    return {
        "question": q_match.group(1).strip() if q_match else "",
        "answer": a_match.group(1).strip() if a_match else ""
    }


def validate_numbers(row, qa, selected_props):

    text = qa["question"] + " " + qa["answer"]

    missing = []

    for p in selected_props:
        val = str(row[p])

        if val not in text:
            missing.append(f"{p}={val}")

    if missing:
        print("[WARNING] missing or modified:", missing)

    return qa


def generate_qa(row, seed=0):
    random.seed(seed)
    torch.manual_seed(seed)       # original did not seed torch -> not reproducible
    k = random.randint(2, 4)
    selected_props = random.sample(properties, k)

    prompt = build_prompt(row, selected_props)

    messages = [
        {
            "role": "system",
            "content": "You are a dataset generator."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    qa = parse_qa(decoded)
    qa = validate_numbers(row, qa, selected_props)

    return {
    "selected_properties": selected_props,

    "question": qa["question"],

    "answer": qa["answer"],

    "property_values": {
        prop: {
            "value": float(row[prop]),
            "unit": units[prop]
        }
        for prop in selected_props
    },

    "num_properties": len(selected_props)
}


for Sample_No, row in df_pred2.iloc[sample_start:sample_end].iterrows():
    results = []
    print (Sample_No)
    for seed in range(500):
        r = generate_qa(row,seed)
        r["sample_id"] = str(Sample_No)      # traceability (not in the original)
        r["seed"] = seed
        results.append(r)

    out_path = os.path.join(OUT_DIR, f"{Sample_No}.jsonl")
    with open(out_path, "w") as f:
        for item in results:
            f.write(json.dumps(item) + "\n")
